# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/melikekaya01/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# Week 4 — Transparent Baseline Action Score

This notebook builds a transparent, rule-based baseline for identifying pages with meaningful CTR improvement opportunities.

The baseline prioritizes pages that already have search visibility but receive a lower-than-expected click-through rate. It is designed as a simple and interpretable decision-support tool rather than a machine learning model.

In [6]:
!git clone https://github.com/melikekaya01/flyrank-ml.git

Cloning into 'flyrank-ml'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 134 (delta 44), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.87 MiB | 6.69 MiB/s, done.
Resolving deltas: 100% (44/44), done.


In [7]:
%cd /content/flyrank-ml

/content/flyrank-ml


In [9]:
from pathlib import Path

print("Current directory:", Path.cwd())
print("data exists:", (Path.cwd() / "data").exists())
print("work exists:", (Path.cwd() / "work").exists())

Current directory: /content/flyrank-ml
data exists: True
work exists: True


In [8]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


In [10]:
# Find the repository root in Colab or a local environment

candidate_roots = [
    Path.cwd(),
    Path("/content/flyrank-ml"),
    Path("/content/flyrank-ml/flyrank-ml"),
]

REPO_ROOT = next(
    (
        path
        for path in candidate_roots
        if (path / "data").exists() and (path / "work").exists()
    ),
    None,
)

if REPO_ROOT is None:
    raise FileNotFoundError(
        "Repository root could not be found. Clone the repository first."
    )

print("Repository root:", REPO_ROOT)

Repository root: /content/flyrank-ml


In [13]:
DATA_PATH = REPO_ROOT / "data" / "raw" / "content_refresh_anonymized.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

print("Dataset path:", DATA_PATH)

Dataset path: /content/flyrank-ml/data/raw/content_refresh_anonymized.csv


In [14]:
df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
print(df.columns.tolist())

display(df.head())

Rows: 30000
Columns: 44

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0000,0.6700,HIGH,2.0500,keyword article,transactional,"3,221.0000","20,457.0000",NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.7600,10.6000,5.8800,4.5500,0.0000,good,striking,down,-41.4000
1,content_a1fb4e703a9e,client_4e07408562,90.0000,0.0100,LOW,0.0500,keyword article,informational,"2,481.0000","15,562.0000",NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.0500,20.3000,0.0000,10.0000,0.0000,good,page_3_5,down,-57.7000
2,content_9aa793d4d895,client_7f2253d7e2,0.0000,0.0000,LOW,0.0000,keyword article,informational,"3,515.0000","23,643.0000",NaN,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.0900,36.5000,0.0000,28.5700,0.0000,good,page_3_5,down,-60.9000
3,content_331d6c4de07b,client_19581e27de,10.0000,0.0000,LOW,0.0000,keyword article,commercial,NaN,NaN,NaN,NaN,11751,58,87,78,75,1,0,3,88,51,3626,22,35,4206,17,26,463,365+,6,22,0-30,NaN,NaN,0.4900,6.2000,1.2800,3.4500,0.0000,good,page_1,stable,-13.8000
4,content_d99b7a2d90ca,client_3fdba35f04,0.0000,0.0000,LOW,0.0000,keyword article,informational,"2,803.0000","17,469.0000",NaN,gemini-3-flash-preview,19140,24,177,145,144,0,0,43,88,33,4211,10,14,6452,2,9,263,181-365,5,14,0-30,2000-3500,15000-25000,0.1300,44.0000,0.0000,24.2900,0.0000,good,page_3_5,down,-34.7000


In [15]:
# Load the dataset

if DATA_PATH.suffix == ".parquet":
    df = pd.read_parquet(DATA_PATH)
elif DATA_PATH.suffix == ".csv":
    df = pd.read_csv(DATA_PATH)
else:
    raise ValueError(f"Unsupported file format: {DATA_PATH.suffix}")

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumn names:")
print(df.columns.tolist())

display(df.head())

Rows: 30000
Columns: 44

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0000,0.6700,HIGH,2.0500,keyword article,transactional,"3,221.0000","20,457.0000",NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.7600,10.6000,5.8800,4.5500,0.0000,good,striking,down,-41.4000
1,content_a1fb4e703a9e,client_4e07408562,90.0000,0.0100,LOW,0.0500,keyword article,informational,"2,481.0000","15,562.0000",NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.0500,20.3000,0.0000,10.0000,0.0000,good,page_3_5,down,-57.7000
2,content_9aa793d4d895,client_7f2253d7e2,0.0000,0.0000,LOW,0.0000,keyword article,informational,"3,515.0000","23,643.0000",NaN,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.0900,36.5000,0.0000,28.5700,0.0000,good,page_3_5,down,-60.9000
3,content_331d6c4de07b,client_19581e27de,10.0000,0.0000,LOW,0.0000,keyword article,commercial,NaN,NaN,NaN,NaN,11751,58,87,78,75,1,0,3,88,51,3626,22,35,4206,17,26,463,365+,6,22,0-30,NaN,NaN,0.4900,6.2000,1.2800,3.4500,0.0000,good,page_1,stable,-13.8000
4,content_d99b7a2d90ca,client_3fdba35f04,0.0000,0.0000,LOW,0.0000,keyword article,informational,"2,803.0000","17,469.0000",NaN,gemini-3-flash-preview,19140,24,177,145,144,0,0,43,88,33,4211,10,14,6452,2,9,263,181-365,5,14,0-30,2000-3500,15000-25000,0.1300,44.0000,0.0000,24.2900,0.0000,good,page_3_5,down,-34.7000


## 1. My rule and its reason codes

The baseline focuses on pages that already receive meaningful search visibility but generate a low click-through rate.

A page enters the action queue when it meets all three conditions:

- At least 500 impressions during the 90-day observation window
- An average search position of 20 or better
- A CTR below 0.5%

These conditions follow the CTR-opportunity definition established in the previous data-contract work. The baseline does not use the proxy-target column directly. Instead, it recreates the business rule from observable, current-window metrics.

The queue uses one transparent reason code and one recommended action:

- **Reason code:** `CTR_OPPORTUNITY_WITH_VISIBILITY`
- **Action label:** `REVIEW_TITLE_AND_META`

In [16]:
# Confirm how CTR is represented in the dataset

ctr_check = df.loc[
    df["impressions_90d"] > 0,
    ["clicks_90d", "impressions_90d", "ctr"],
].copy()

ctr_check["recalculated_ctr_pct"] = (
    ctr_check["clicks_90d"]
    / ctr_check["impressions_90d"]
    * 100
)

ctr_check["absolute_difference"] = (
    ctr_check["ctr"] - ctr_check["recalculated_ctr_pct"]
).abs()

display(ctr_check.head())

print(
    "Median absolute CTR difference:",
    ctr_check["absolute_difference"].median(),
)

,clicks_90d,impressions_90d,ctr,recalculated_ctr_pct,absolute_difference
0,29,3803,0.7600,0.7626,0.0026
1,7,15320,0.0500,0.0457,0.0043
2,11,12581,0.0900,0.0874,0.0026
3,58,11751,0.4900,0.4936,0.0036
4,24,19140,0.1300,0.1254,0.0046


Median absolute CTR difference: 0.0004916606650075678


In [17]:
# Apply the transparent CTR-opportunity eligibility rule

eligibility_mask = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)

eligible_pages = df.loc[eligibility_mask].copy()

print("Total dataset rows:", len(df))
print("Eligible queue rows:", len(eligible_pages))
print(
    "Eligible share:",
    f"{len(eligible_pages) / len(df) * 100:.2f}%",
)

display(
    eligible_pages[
        [
            "content_id",
            "client_id",
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "avg_position",
            "engagement_rate",
        ]
    ].head(10)
)

Total dataset rows: 30000
Eligible queue rows: 9759
Eligible share: 32.53%


,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate
3,content_331d6c4de07b,client_19581e27de,11751,58,0.4900,6.2000,1.2800
5,content_d4084a4bc775,client_f369cb89fc,3970,1,0.0300,8.5000,0.0000
9,content_c27558df2b0c,client_19581e27de,1240,2,0.1600,4.9000,0.0000
16,content_78bd1d4a1d4d,client_6208ef0f77,13848,21,0.1500,8.9000,0.2100
17,content_761a44afda12,client_19581e27de,9449,7,0.0700,7.3000,11.8600
18,content_0b360eb9db55,client_349c41201b,5141,7,0.1400,11.4000,2.3300
22,content_3fb46bec4413,client_4e07408562,2311,3,0.1300,3.9000,0.0000
24,content_0e23e310d404,client_19581e27de,29541,79,0.2700,4.1000,0.9300
27,content_7ea135180dd9,client_4ec9599fc2,1197,2,0.1700,8.1000,0.0000
34,content_55f75c034970,client_d029fa3a95,3998,1,0.0300,6.4000,0.0000


## 2. Build the ranked queue

The eligibility rule identifies pages with an observable CTR opportunity. The action score then ranks those pages by practical review priority.

The score uses four interpretable components:

- **Visibility score:** higher impressions increase priority
- **Position score:** stronger rankings increase priority because better-ranked pages have greater click potential
- **Low-CTR score:** lower CTR increases priority
- **Low-engagement score:** lower engagement adds a smaller secondary signal

The score is a prioritization heuristic, not a probability or causal estimate.

In [18]:

# Create transparent score components

queue = eligible_pages.copy()

# 1. Visibility score: percentile rank of impression volume
queue["visibility_score"] = (
    queue["impressions_90d"]
    .rank(pct=True)
    .mul(100)
)

# 2. Position score: better positions receive higher scores
queue["position_score"] = (
    (20 - queue["avg_position"]) / 19 * 100
).clip(lower=0, upper=100)

# 3. Low-CTR score: CTR closer to zero receives higher priority
queue["low_ctr_score"] = (
    (0.5 - queue["ctr"]) / 0.5 * 100
).clip(lower=0, upper=100)

# 4. Low-engagement score:
# lower engagement receives higher priority within the eligible set
queue["low_engagement_score"] = (
    100
    - queue["engagement_rate"].rank(pct=True).mul(100)
)

# Weighted transparent action score
queue["action_score"] = (
    0.40 * queue["visibility_score"]
    + 0.30 * queue["position_score"]
    + 0.20 * queue["low_ctr_score"]
    + 0.10 * queue["low_engagement_score"]
)

queue["reason_code"] = "CTR_OPPORTUNITY_WITH_VISIBILITY"
queue["action_label"] = "REVIEW_TITLE_AND_META"

queue = (
    queue
    .sort_values(
        by=[
            "action_score",
            "impressions_90d",
            "avg_position",
        ],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

queue.insert(0, "rank", range(1, len(queue) + 1))

display(
    queue[
        [
            "rank",
            "content_id",
            "client_id",
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "avg_position",
            "engagement_rate",
            "visibility_score",
            "position_score",
            "low_ctr_score",
            "low_engagement_score",
            "action_score",
            "reason_code",
            "action_label",
        ]
    ].head(20)
)

print("Queue rows:", len(queue))
print("Maximum action score:", queue["action_score"].max())
print("Minimum action score:", queue["action_score"].min())

,rank,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,visibility_score,position_score,low_ctr_score,low_engagement_score,action_score,reason_code,action_label
0,1,content_0022a6b4290f,client_f369cb89fc,29747,22,0.0700,1.2000,0.0000,93.1858,98.9474,86.0000,70.5042,91.2089,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
1,2,content_339b357d04c7,client_bbb965ab0c,46879,7,0.0100,3.7000,0.0000,96.5468,85.7895,98.0000,70.5042,91.0060,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
2,3,content_4a6607efcb46,client_6208ef0f77,128068,17,0.0100,2.2000,2.3000,99.4262,93.6842,98.0000,31.9551,90.6712,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
3,4,content_f4e210ee0c27,client_7f2253d7e2,24784,16,0.0600,1.6000,0.0000,91.5360,96.8421,88.0000,70.5042,90.3175,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
4,5,content_8451fc6f034d,client_d029fa3a95,272144,75,0.0300,2.3000,2.0200,99.9180,93.1579,94.0000,33.5639,90.0710,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
5,6,content_fea6a0d13b4a,client_19581e27de,79965,56,0.0700,3.4000,0.0000,98.5039,87.3684,86.0000,70.5042,89.8625,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
6,7,content_954cc45bd437,client_4e07408562,15439,6,0.0400,1.5000,0.0000,85.9002,97.3684,92.0000,70.5042,89.0210,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
7,8,content_f9d82e71e363,client_19581e27de,24260,19,0.0800,2.0000,0.0000,91.3003,94.7368,84.0000,70.5042,88.7916,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
8,9,content_dd635253d90e,client_6208ef0f77,33286,5,0.0200,4.6000,0.0000,94.2412,81.0526,96.0000,70.5042,88.2627,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
9,10,content_cbdf5a78dcd0,client_19581e27de,14830,3,0.0200,2.4000,0.0000,85.3161,92.6316,96.0000,70.5042,88.1663,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META


Queue rows: 9759
Maximum action score: 91.20893642036232
Minimum action score: 6.98813295149956


## 3. Top-20 review

The first 20 rows are consistent with the intended business logic.

These pages combine:

- strong search visibility,
- rankings that are already close to the top of the search results,
- very low CTR,
- and, in many cases, weak engagement.

For example, the highest-ranked pages receive tens of thousands of impressions while maintaining CTR values below 0.10%. This creates a clear optimization opportunity because even a small improvement in title or meta-description performance could translate into additional clicks.

The ranking is also interpretable. Pages do not rise to the top because of one hidden model prediction. Their priority comes from observable components: impression volume, average position, CTR, and engagement.

The results should still be reviewed by a human before action is taken. Low CTR can also be influenced by search intent, SERP features, brand visibility, or a mismatch between the query and the page.

In [19]:
top_20 = queue.head(20).copy()

display(
    top_20[
        [
            "rank",
            "content_id",
            "client_id",
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "avg_position",
            "engagement_rate",
            "action_score",
            "reason_code",
            "action_label",
        ]
    ]
)

,rank,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,action_score,reason_code,action_label
0,1,content_0022a6b4290f,client_f369cb89fc,29747,22,0.0700,1.2000,0.0000,91.2089,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
1,2,content_339b357d04c7,client_bbb965ab0c,46879,7,0.0100,3.7000,0.0000,91.0060,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
2,3,content_4a6607efcb46,client_6208ef0f77,128068,17,0.0100,2.2000,2.3000,90.6712,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
3,4,content_f4e210ee0c27,client_7f2253d7e2,24784,16,0.0600,1.6000,0.0000,90.3175,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
4,5,content_8451fc6f034d,client_d029fa3a95,272144,75,0.0300,2.3000,2.0200,90.0710,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
5,6,content_fea6a0d13b4a,client_19581e27de,79965,56,0.0700,3.4000,0.0000,89.8625,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
6,7,content_954cc45bd437,client_4e07408562,15439,6,0.0400,1.5000,0.0000,89.0210,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
7,8,content_f9d82e71e363,client_19581e27de,24260,19,0.0800,2.0000,0.0000,88.7916,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
8,9,content_dd635253d90e,client_6208ef0f77,33286,5,0.0200,4.6000,0.0000,88.2627,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
9,10,content_cbdf5a78dcd0,client_19581e27de,14830,3,0.0200,2.4000,0.0000,88.1663,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META


## 4. Weak picks + leakage check

The bottom-ranked pages still satisfy the eligibility rule, but they represent weaker opportunities.

Compared with the highest-ranked pages, these records generally have:

- lower impression volume,
- positions closer to the eligibility boundary,
- CTR values closer to 0.5%,
- or stronger engagement.

This confirms that the action score is functioning as a ranking mechanism rather than treating every eligible page as equally urgent.

The score does not use:

- `trend_direction`,
- `trend_pct`,
- any future-period metric,
- or a precomputed proxy-target column.

Although CTR is related to the Week 3 proxy definition, it is used here as an observable business-rule input rather than as a machine-learning feature. The baseline is therefore a transparent rule system, not a trained predictive model.

In [20]:
weak_picks = queue.tail(5).copy()

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "client_id",
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "avg_position",
            "engagement_rate",
            "action_score",
        ]
    ]
)

,rank,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,action_score
9754,9755,content_65f8f6c0865b,client_6208ef0f77,736,3,0.4100,18.3000,2.9400,12.9099
9755,9756,content_ee3833c68977,client_3fdba35f04,1030,5,0.4900,18.1000,21.4300,10.9366
9756,9757,content_115a5fc92151,client_a88a7902cb,626,3,0.4800,19.5000,0.0000,10.6442
9757,9758,content_896b4b3c39f7,client_4e07408562,508,2,0.3900,17.1000,20.0000,9.3965
9758,9759,content_ace104f58973,client_4e07408562,563,2,0.3600,19.8000,50.0000,6.9881


## 5. Final output

The final output keeps only the fields required for review and removes diagnostic or leakage-sensitive columns that are not needed in the delivered action queue.

The CSV is ordered from the highest-priority CTR opportunity to the lowest-priority eligible page.

In [21]:
safe_queue_columns = [
    "rank",
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "visibility_score",
    "position_score",
    "low_ctr_score",
    "low_engagement_score",
    "action_score",
    "reason_code",
    "action_label",
]

final_queue = queue[safe_queue_columns].copy()

OUTPUT_DIR = REPO_ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "baseline_action_score.csv"

final_queue.to_csv(OUTPUT_PATH, index=False)

print("Output saved:", OUTPUT_PATH)
print("Output rows:", len(final_queue))

display(final_queue.head())

Output saved: /content/flyrank-ml/work/outputs/baseline_action_score.csv
Output rows: 9759


,rank,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,visibility_score,position_score,low_ctr_score,low_engagement_score,action_score,reason_code,action_label
0,1,content_0022a6b4290f,client_f369cb89fc,29747,22,0.0700,1.2000,0.0000,93.1858,98.9474,86.0000,70.5042,91.2089,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
1,2,content_339b357d04c7,client_bbb965ab0c,46879,7,0.0100,3.7000,0.0000,96.5468,85.7895,98.0000,70.5042,91.0060,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
2,3,content_4a6607efcb46,client_6208ef0f77,128068,17,0.0100,2.2000,2.3000,99.4262,93.6842,98.0000,31.9551,90.6712,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
3,4,content_f4e210ee0c27,client_7f2253d7e2,24784,16,0.0600,1.6000,0.0000,91.5360,96.8421,88.0000,70.5042,90.3175,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META
4,5,content_8451fc6f034d,client_d029fa3a95,272144,75,0.0300,2.3000,2.0200,99.9180,93.1579,94.0000,33.5639,90.0710,CTR_OPPORTUNITY_WITH_VISIBILITY,REVIEW_TITLE_AND_META


## 6. Self-check

The checks below confirm that the queue contains the required fields, is correctly ranked, excludes future-window and target-derived columns, and can regenerate the required CSV.

In [22]:
required_columns = {
    "rank",
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "action_score",
    "reason_code",
    "action_label",
}

missing_columns = required_columns.difference(final_queue.columns)

assert not missing_columns, (
    f"Missing required columns: {sorted(missing_columns)}"
)

assert len(final_queue) == 9759
assert len(final_queue) > 0

assert final_queue["action_score"].is_monotonic_decreasing

assert final_queue["rank"].tolist() == list(
    range(1, len(final_queue) + 1)
)

assert final_queue["reason_code"].nunique() == 1
assert (
    final_queue["reason_code"].iloc[0]
    == "CTR_OPPORTUNITY_WITH_VISIBILITY"
)

assert final_queue["action_label"].nunique() == 1
assert (
    final_queue["action_label"].iloc[0]
    == "REVIEW_TITLE_AND_META"
)

for forbidden_column in [
    "trend_direction",
    "trend_pct",
    "ctr_opportunity_proxy",
]:
    assert forbidden_column not in final_queue.columns

assert OUTPUT_PATH.exists()

print("Self-check passed.")
print("Final queue rows:", len(final_queue))
print(
    "Reason code:",
    final_queue["reason_code"].unique().tolist(),
)
print(
    "Action label:",
    final_queue["action_label"].unique().tolist(),
)
print("Output file:", OUTPUT_PATH)

Self-check passed.
Final queue rows: 9759
Reason code: ['CTR_OPPORTUNITY_WITH_VISIBILITY']
Action label: ['REVIEW_TITLE_AND_META']
Output file: /content/flyrank-ml/work/outputs/baseline_action_score.csv


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.